# Medical LLM Fine-Tuning: Data Exploration

In this notebook, we will install our necessary dependencies, load our initial medical Q&A dataset, and perform exploratory data analysis. This helps us understand our data quality before we proceed to fine-tuning.

> **Environment Note:** This notebook is optimised to run on **Google Colab**. All output files are saved to Google Drive so they persist across sessions and are available to downstream notebooks.

## Cell 1 — Install Dependencies

We install the core libraries needed for loading the HuggingFace dataset, data manipulation, and visualization.

In [ ]:
# Install the necessary libraries for data manipulation, visualization, and huggingface datasets
!pip install datasets transformers pandas matplotlib seaborn

## Cell 2 — Mount Google Drive

Colab's local filesystem is **ephemeral** — every time the runtime disconnects, all files are wiped. By mounting Google Drive, we persist all output files (CSVs, charts) across sessions so that notebook `02_baseline_evaluation.ipynb` can read them directly without any manual re-uploads.

All project files will live under `/content/drive/MyDrive/medical-llm/`.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive — a browser popup will ask you to authorise access
drive.mount('/content/drive')

# Define the shared project directory on Drive and create it if it doesn't exist
DRIVE_DIR = '/content/drive/MyDrive/medical-llm'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Google Drive mounted. Project files will be saved to: {DRIVE_DIR}")

## Cell 3 — Load the MedQuAD Dataset

We will use the `lavita/medical-qa-datasets` dataset from Hugging Face. This dataset contains medical question-answer pairs which are ideal for our fine-tuning task. We'll print the dataset structure to understand the available splits and features.

In [ ]:
from datasets import load_dataset

# Load the medical Q&A dataset from HuggingFace Hub
dataset = load_dataset("lavita/medical-qa-datasets")

# Print the dataset object to view its structure, splits, and features
print("Dataset Structure:")
print(dataset)

## Cell 4 — Basic Statistics and Data Inspection

Here we'll calculate the total number of samples and inspect the column names. We'll also print 5 random examples to get a feel for the data format, and calculate the average character length of questions and answers.

In [ ]:
import random
import numpy as np

# We'll use the 'train' split for exploration
train_data = dataset['train']

print(f"Total number of samples: {len(train_data)}")
print(f"Column names: {train_data.column_names}")

print("\n--- 5 Random Examples ---")
for i in range(5):
    idx = random.randint(0, len(train_data) - 1)
    sample = train_data[idx]
    print(f"\nExample {i+1}:")
    # Using .get in case the columns are named differently (e.g., 'input' vs 'question')
    print(f"Q: {sample.get('question', sample.get('input', 'N/A'))}")
    print(f"A: {sample.get('answer', sample.get('output', 'N/A'))}")

# Dynamically identify question and answer columns based on dataset schema
q_col = 'question' if 'question' in train_data.column_names else 'input'
a_col = 'answer'   if 'answer'   in train_data.column_names else 'output'

# Calculate lengths, filtering out None/Null values before measuring
q_lengths = [len(str(x)) for x in train_data[q_col] if x is not None]
a_lengths = [len(str(x)) for x in train_data[a_col] if x is not None]

print("\n--- Length Statistics ---")
print(f"Average Question Length: {np.mean(q_lengths):.2f} characters")
print(f"Average Answer Length:   {np.mean(a_lengths):.2f} characters")

## Cell 5 — Quality Filtering

Real-world datasets often contain noise or incomplete entries. We will filter out:
- Samples where the answer is missing, an empty string, or very short (< 50 chars).
- Samples where the question is too short (< 10 chars).

This ensures our model learns from high-quality, substantive medical answers.

In [ ]:
# Define our filtering logic
def is_valid_sample(example):
    q = example.get(q_col, "")
    a = example.get(a_col, "")

    # Reject if answer is None or too short to be meaningful
    if a is None or len(str(a).strip()) < 50:
        return False

    # Reject if question is None or too short
    if q is None or len(str(q).strip()) < 10:
        return False

    return True

# Apply the filter function to the dataset
original_count = len(train_data)
filtered_data  = train_data.filter(is_valid_sample)
new_count      = len(filtered_data)

removed_count      = original_count - new_count
removed_percentage = (removed_count / original_count) * 100

print(f"Samples remaining after filtering: {new_count}")
print(f"Samples removed: {removed_count} ({removed_percentage:.2f}%)")

## Cell 6 — Category Analysis

Understanding the distribution of topics or categories within our dataset helps identify any biases. We will plot the top 15 most frequent categories (if available) and save this chart to Google Drive.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Convert filtered dataset to pandas for easier analysis
df = filtered_data.to_pandas()

# Search for a column that likely represents categories or topics
cat_col = None
potential_cat_cols = ['category', 'topic', 'source', 'focus_area', 'qtype']
for col in potential_cat_cols:
    if col in df.columns:
        cat_col = col
        break

if cat_col:
    print(f"Found category column: '{cat_col}'")
    plt.figure(figsize=(12, 6))

    top_categories = df[cat_col].value_counts().nlargest(15)

    sns.barplot(x=top_categories.values, y=top_categories.index, palette='viridis')
    plt.title(f"Top 15 {cat_col.capitalize()}s by Count")
    plt.xlabel("Count")
    plt.ylabel(cat_col.capitalize())
    plt.tight_layout()

    # Save to Google Drive for persistence
    chart_path = f'{DRIVE_DIR}/category_distribution.png'
    plt.savefig(chart_path)
    plt.show()
    print(f"Chart saved to: {chart_path}")
else:
    print("No prominent category or topic column found in the dataset schema.")

## Cell 7 — Length Distribution Analysis

To determine the optimal maximum token length for our fine-tuning process, we will visualize the distribution of answer lengths in characters. We'll add reference lines at 500 and 2000 characters to gauge how many samples might get truncated.

In [ ]:
plt.figure(figsize=(10, 6))

# Extract lengths of all filtered answers
ans_lengths = df[a_col].astype(str).apply(len)

# Plot histogram
sns.histplot(ans_lengths, bins=50, color='skyblue', edgecolor='black')
plt.axvline(x=500,  color='red',   linestyle='--', label='500 Chars')
plt.axvline(x=2000, color='green', linestyle='--', label='2000 Chars')

plt.title("Distribution of Answer Lengths (Characters)")
plt.xlabel("Answer Length")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()

# Save to Google Drive for persistence
dist_path = f'{DRIVE_DIR}/answer_length_distribution.png'
plt.savefig(dist_path)
plt.show()
print(f"Chart saved to: {dist_path}")

## Cell 8 — Save the Cleaned Dataset to Google Drive

Finally, we save our cleaned and filtered dataset to Google Drive as `cleaned_medquad.csv`. Saving to Drive (rather than Colab's local `/content/`) ensures this file survives session disconnects and is immediately available to notebook `02_baseline_evaluation.ipynb` without any manual re-upload.

In [ ]:
# Keep only the question and answer columns and rename them to standard names
final_df = df[[q_col, a_col]].rename(columns={q_col: 'question', a_col: 'answer'})

# Save to Google Drive — this file will be read directly by notebook 02
output_path = f'{DRIVE_DIR}/cleaned_medquad.csv'
final_df.to_csv(output_path, index=False)

print(f"Successfully saved {len(final_df)} final samples to:")
print(f"  {output_path}")
print("\nFinal Dataset Preview:")
print(final_df.head(3))